# 03. Experimentación y Selección de Modelos

Con los datos limpios, enriquecidos y escalados, es hora de encontrar el algoritmo predictivo ganador.

### Instrucciones Generales:
1. **Validación:** No entrenes y midas sobre el mismo conjunto (sobreajuste). Recuerda haber dividido en Entrenamiento y Prueba antes.
2. **Entrenamiento Base:** Entrena los siguientes modelos base con tu set de Entrenamiento y compáralos usando RMSE (Error Cuadrático Medio):
   - `LinearRegression`
   - `SGDRegressor`
   - `DecisionTreeRegressor`
   - `RandomForestRegressor`
3. **Cross Validation (Validación Cruzada):** Para tener una métrica robusta, usa `cross_val_score` en el set de Entrenamiento para cada uno de los modelos anteriores.
4. **Ajuste Fino (Fine Tuning):** Toma el modelo ganador y busca sus mejores hiperparámetros. Utiliza un `GridSearchCV` explorando el número de estimadores (`n_estimators`), las características máximas (`max_features`), etc.
5. **Conclusión y Benchmark (IMPORTANTE):** Redacta una conclusión comparando los algoritmos. Explica por qué escogiste el modelo final y valida tu decisión calculando el RMSE sobre tu Set de Prueba que habías reservado. Documenta si alguno de tus modelos se sobreajusto o subajusto. Recuerda que el modelo final no puede tener esos problemas!


In [1]:
# Empieza importando los algoritmos desde Scikit-Learn
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import (
  mean_squared_error,
  mean_absolute_error,
  r2_score
)

import numpy as np
from math import sqrt

In [2]:
def load_dataset(path, target_column):
  """
    Loads a dataset from a given path. Separates the target column from the features.
  """
  df = pd.read_csv(path)
  X = df.drop(columns=[target_column])
  y = df[target_column]
  return X, y

def split_data(data_path, test_size):
  df = pd.read_csv(data_path)
  return train_test_split(df, test_size=test_size, random_state=42)

def separate_features(data, target_column):
  X = data.drop(target_column, axis=1)
  y = data[target_column]
  return X, y

def drop_columns(data, columns, target_column):
  X1 = data.drop(columns, axis=1)
  X = X1.drop(target_column, axis=1)
  y = data[target_column]
  return X, y

def print_metrics(model, X_values, y_eval):
  y_pred = model.predict(X_values)
  mae = mean_absolute_error(y_eval, y_pred)
  rmse = sqrt(mean_squared_error(y_eval, y_pred))
  r2 = r2_score(y_eval, y_pred)
  print(f"MAE: {mae}")
  print(f"RMSE: {rmse}")
  print(f"R2: {r2}")
  
def just_print_metrics(y_pred, y_true):
  mae = mean_absolute_error(y_true, y_pred)
  rmse = sqrt(mean_squared_error(y_true, y_pred))
  r2 = r2_score(y_true, y_pred)
  print(f"MAE: {mae}")
  print(f"RMSE: {rmse}")
  print(f"R2: {r2}")

def print_metrics_values(x, y):
  mae = mean_absolute_error(x, y)
  rmse = sqrt(mean_squared_error(x, y))
  r2 = r2_score(x, y_test)
  print(f"MAE: {mae}")
  print(f"RMSE: {rmse}")
  print(f"R2: {r2}")

In [3]:
interim_data_path = '../data/interim'
target = 'median_house_value'
train, test = split_data('../data/processed/processed_data.csv', 0.2)
X_train, y_train = separate_features(train, target)
X_test, y_test = separate_features(test, target)

In [ ]:
rf = RandomForestRegressor()
rf.fit(X_train, y_train)

In [ ]:
print_metrics(rf, X_test, y_test)

In [ ]:
param_grid = {
  'n_estimators': [100, 200],
  'learning_rate': [0.05, 0.1],
  'max_depth': [3, 5],
  'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
  GradientBoostingRegressor(random_state=42),
  param_grid,
  cv=5,
  scoring='neg_root_mean_squared_error',
  n_jobs=-1,
  verbose=2
)

grid_search.fit(X_train, y_train)

In [ ]:
param_grid = {
  'n_estimators': [100, 200],
  'max_depth': [10, 20, None],
  'min_samples_split': [2, 5],
  'min_samples_leaf': [1, 2]
}

grid_search = GridSearchCV(
  estimator=RandomForestRegressor(),
  param_grid=param_grid,
  cv=5,
  scoring='neg_root_mean_squared_error',  # best for regression
  n_jobs=-1,
  verbose=2
)

# Fit
grid_search.fit(X_train, y_train)

# Best results
print("Best params:", grid_search.best_params_)
print("Best RMSE:", -grid_search.best_score_)

In [ ]:
best_model = grid_search.best_estimator_
print('Metricas para testeo')
print_metrics(best_model, X_test, y_test)

In [ ]:
importances = rf.feature_importances_

feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(feature_importance)

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

param_grid = {
  'n_estimators': [100, 200],
  'learning_rate': [0.05, 0.1],
  'max_depth': [3, 5],
  'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
  GradientBoostingRegressor(random_state=42),
  param_grid,
  cv=5,
  scoring='neg_root_mean_squared_error',
  n_jobs=-1,
  verbose=2
)

grid_search.fit(X_train, y_train)

In [ ]:
best_gbr = grid_search.best_estimator_
print('Metricas para testeo')
print_metrics(best_gbr, X_test, y_test)

In [ ]:
from xgboost import XGBRegressor

param_grid = {
    'n_estimators': [200, 500],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 6],
    'subsample': [0.8, 1.0]
}

grid_search = GridSearchCV(
    XGBRegressor(random_state=42),
    param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

In [ ]:
best_xgbr = grid_search.best_estimator_
print('Metricas para testeo')
print_metrics(best_xgbr, X_test, y_test)

In [ ]:
X_train, y_train = drop_columns(train, ['total_rooms', 'total_bedrooms', 'income_per_person', 'households', 'population'], target)
X_test, y_test = drop_columns(test, ['total_rooms', 'total_bedrooms', 'income_per_person', 'households', 'population'], target)

In [ ]:
model = RandomForestRegressor(max_depth=25, min_samples_leaf=2, min_samples_split=5, n_estimators=300)
y_train_log = np.log1p(y_train)
model.fit(X_train, y_train_log)

In [ ]:
y_test_log = np.log1p(y_test)
print_metrics(model, X_test, y_test_log)

In [ ]:
from xgboost import XGBRegressor

param_grid = {
    'n_estimators': [200, 500],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 6],
    'subsample': [0.8, 1.0]
}

grid_search = GridSearchCV(
    XGBRegressor(random_state=42),
    param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, np.log1p(y_train))

In [ ]:
best_xgbr_1 = grid_search.best_estimator_
y_test_log = np.log1p(y_test)
print_metrics(best_xgbr_1, X_test, y_test_log)

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

mlp = MLPRegressor(hidden_layer_sizes=(64,32,16), max_iter=5000)
scaler = StandardScaler()
scaler_y = StandardScaler()


In [ ]:
X_scaled = scaler.fit_transform(X_train)
y_train_2d = y_train.to_numpy().reshape(-1, 1)
y_scaled = scaler_y.fit_transform(y_train_2d)
mlp.fit(X_scaled, y_scaled.ravel())

In [ ]:
X_scaled = scaler.transform(X_test)
y_ans = mlp.predict(X_scaled)
y_pred = scaler_y.inverse_transform(y_ans.reshape(-1, 1)).ravel()


In [ ]:
print_metrics_values(y_pred, y_test)

In [5]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV

In [16]:
param_grid = {
  "n_estimators": [300, 500],
  "max_depth": [6, 7],
  "learning_rate": [0.1],
  "subsample": [1.0],
  "colsample_bytree": [0.8, 0.7],
  "gamma": [0, 0.02],
  "reg_alpha": [1],
  "reg_lambda": [5, 10]
}

xgb = XGBRegressor(
  objective='reg:squarederror',
  random_state=42
)

grid_XGB = GridSearchCV(
  estimator=xgb,
  param_grid=param_grid,
  scoring='neg_root_mean_squared_error',
  cv=3,
  verbose=2,
  n_jobs=-1
)

grid_XGB.fit(X_train, y_train)

Fitting 3 folds for each of 32 candidates, totalling 96 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'colsample_bytree': [0.8, 0.7], 'gamma': [0, 0.02], 'learning_rate': [0.1], 'max_depth': [6, 7], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and par

In [17]:
print_metrics(grid_XGB.best_estimator_, X_train, y_train)

MAE: 11236.583303780359
RMSE: 16718.649076718597
R2: 0.9789382350543226


In [18]:
print_metrics(grid_XGB.best_estimator_, X_test, y_test)

MAE: 28915.093502454427
RMSE: 44584.204474750266
R2: 0.8531679005489855


In [ ]:
best_model = grid_XGB.best_estimator_

print("Best params:", grid.best_params_)
print("Best RMSE:", -grid_XGB.best_score_)

Best params: {'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 500, 'reg_alpha': 1, 'reg_lambda': 10, 'subsample': 1.0}
Best RMSE: 45477.22436545081


In [6]:
xgb = XGBRegressor(
  objective='reg:squarederror',

  max_depth=6,
  learning_rate=0.03,
  n_estimators=1000,

  subsample=0.8,
  colsample_bytree=0.7,

  gamma=0.03,
  reg_alpha=1.0,
  reg_lambda=10.0,

  random_state=42,
  n_jobs=-1
)

In [7]:
xgb.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.7
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [8]:
y_pred = xgb.predict(X_test)
just_print_metrics(y_true=y_test, y_pred=y_pred)

MAE: 29268.768781922867
RMSE: 44558.17426342438
R2: 0.8533393045193847


In [9]:
y_pred = xgb.predict(X_train)
just_print_metrics(y_true=y_train, y_pred=y_pred)

MAE: 19556.685700176076
RMSE: 28546.584860731287
R2: 0.9385954269263032


In [11]:

import joblib
joblib.dump(xgb, 'best')

['best']

In [40]:
print('train')
print_metrics(xgb, X_train, y_train)
print('test')
print_metrics(xgb, X_test, y_test)

train
MAE: 19556.685700176076
RMSE: 28546.584860731287
R2: 0.9385954269263032
test
MAE: 29268.768781922867
RMSE: 44558.17426342438
R2: 0.8533393045193847


In [7]:
param_grid = {
  "n_estimators": [300, 500],
  "max_depth": [30, 20],
  "min_samples_split": [2],
  "min_samples_leaf": [2],
  "max_features": ["log2"],
}

rfr = RandomForestRegressor()

grid = GridSearchCV(
  estimator=rfr,
  param_grid=param_grid,
  scoring='neg_root_mean_squared_error',
  cv=3,
  verbose=2,
  n_jobs=-1
)

grid.fit(X_train, y_train)

Fitting 3 folds for each of 4 candidates, totalling 12 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestRegressor()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [30, 20], 'max_features': ['log2'], 'min_samples_leaf': [2], 'min_samples_split': [2], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter

In [ ]:
print_metrics(grid.best_estimator_, X_train, y_train)

MAE: 11236.583303780359
RMSE: 16718.649076718597
R2: 0.9789382350543226


In [8]:
best_model_rf = grid.best_estimator_

print("Best params:", grid.best_params_)
print("Best RMSE:", -grid.best_score_)

Best params: {'max_depth': 30, 'max_features': 'log2', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 500}
Best RMSE: 52276.69273927316


### Benchmark y Conclusión Final
*(Escribe tus conclusiones de negocio aquí)*

Se hará una evaluación de varios modelos, se tomará el mejor y se le pasará por un grid searach para escoger el mejor tuneado.
Esto me permitirá evaluar varios modelos a la vez
XGBoost se postula como el mejor candidato hasta ahora
Tiene estas métricas en test
MAE: 29268.768781922867
RMSE: 44558.17426342438
R2: 0.8533393045193847